In [ ]:
import os
import time
import pickle
import math
import numpy as np
import pandas as pd

from math import sqrt
from numpy import array

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae
from sklearn.metrics import r2_score
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from scipy.stats import wilcoxon

from statsmodels.tsa.arima.model import ARIMA

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, GRU, SimpleRNN, Bidirectional, Flatten, Dropout
from tensorflow.keras.optimizers import Adam

In [ ]:


# =========================================================
# GLOBALS
# =========================================================
time_results = []
target_scaler = None


# =========================================================
# DATA PREP
# =========================================================
def to_supervised(X, y, n_input, n_output):
    Xs, ys = [], []
    for i in range(len(X) - n_input - n_output + 1):
        Xs.append(X.iloc[i:i+n_input].values)
        ys.append(y.iloc[i+n_input:i+n_input+n_output].values)
    return np.array(Xs), np.array(ys)


def true_y_to_evaluation_form(y_true, n_input, n_out):
    y = []
    for i in range(len(y_true) - n_input - n_out + 1):
        y.append(y_true.iloc[i+n_input:i+n_input+n_out].values)
    return np.array(y)


def prepare_company_for_training_and_testing(company,
                                             selected_ti,
                                             target_col='close',
                                             n_input=5,
                                             n_output=5):

    train = company.loc[:'2015-12-31']
    val   = company.loc['2016-01-01':'2016-12-31']
    test  = company.loc['2017-01-01':]

    print(f'Train shape: {train.shape}, Val shape: {val.shape}, Test shape: {test.shape}')

    feature_scaler = StandardScaler()
    feature_scaler.fit(train[selected_ti])

    train_x_scaled = pd.DataFrame(
        feature_scaler.transform(train[selected_ti]),
        columns=selected_ti,
        index=train.index
    )
    val_x_scaled = pd.DataFrame(
        feature_scaler.transform(val[selected_ti]),
        columns=selected_ti,
        index=val.index
    )
    test_x_scaled = pd.DataFrame(
        feature_scaler.transform(test[selected_ti]),
        columns=selected_ti,
        index=test.index
    )

    global target_scaler
    target_scaler = StandardScaler()
    target_scaler.fit(train[[target_col]])

    train_y_scaled = pd.DataFrame(
        target_scaler.transform(train[[target_col]]),
        columns=[target_col],
        index=train.index
    )
    val_y_scaled = pd.DataFrame(
        target_scaler.transform(val[[target_col]]),
        columns=[target_col],
        index=val.index
    )
    test_y_scaled = pd.DataFrame(
        target_scaler.transform(test[[target_col]]),
        columns=[target_col],
        index=test.index
    )

    train_x, train_y = to_supervised(train_x_scaled, train_y_scaled, n_input, n_output)
    val_x, val_y = to_supervised(val_x_scaled, val_y_scaled, n_input, n_output)
    test_x, test_y = to_supervised(test_x_scaled, test_y_scaled, n_input, n_output)

    return train_x, train_y, val_x, val_y, test_x, test_y, test

def prepare_company_for_training_and_testing_base(company, selected_ti,
                                            target_col='close',
                                            n_input=5,
                                            n_output=5):

    train = company.loc[:'2015-12-31']
    val   = company.loc['2016-01-01':'2016-12-31']
    test  = company.loc['2017-01-01':]

    feature_scaler = StandardScaler()
    feature_scaler.fit(train[selected_ti])

    train_x_scaled = pd.DataFrame(
        feature_scaler.transform(train[selected_ti]),
        columns=selected_ti,
        index=train.index
    )

    val_x_scaled = pd.DataFrame(
        feature_scaler.transform(val[selected_ti]),
        columns=selected_ti,
        index=val.index
    )

    test_x_scaled = pd.DataFrame(
        feature_scaler.transform(test[selected_ti]),
        columns=selected_ti,
        index=test.index
    )

    global target_scaler
    target_scaler = StandardScaler()
    target_scaler.fit(train[[target_col]])

    train_y_scaled = pd.DataFrame(
        target_scaler.transform(train[[target_col]]),
        columns=[target_col],
        index=train.index
    )

    val_y_scaled = pd.DataFrame(
        target_scaler.transform(val[[target_col]]),
        columns=[target_col],
        index=val.index
    )

    test_y_scaled = pd.DataFrame(
        target_scaler.transform(test[[target_col]]),
        columns=[target_col],
        index=test.index
    )

    train_x, train_y = to_supervised(train_x_scaled, train_y_scaled, n_input, n_output)
    val_x, val_y = to_supervised(val_x_scaled, val_y_scaled, n_input, n_output)
    test_x, test_y = to_supervised(test_x_scaled, test_y_scaled, n_input, n_output)

    return train_x, train_y, val_x, val_y, test_x, test_y, train, val, test


# =========================================================
# MRE FUNCTION
# =========================================================
def mre(actual, predicted):

    actual = np.array(actual)
    predicted = np.array(predicted)

    epsilon = 1e-8   # avoid division by zero

    relative_errors = np.abs((actual - predicted) / (actual + epsilon))

    return np.mean(relative_errors)

def evaluate_forecasts(actual, predicted_scaled_or_original, actual_are_scaled = False ,predictions_are_scaled=True):
    if actual_are_scaled:
        actual_original = target_scaler.inverse_transform(np.squeeze(actual))
    else:
        actual_original = np.squeeze(actual)

    predicted = np.squeeze(predicted_scaled_or_original)

    # actual always inverse-transformed because actual passed from pipeline is scaled


    if predictions_are_scaled:
        predicted_original = target_scaler.inverse_transform(predicted)
    else:
        predicted_original = predicted

    actual_original = np.round(actual_original, 2)
    predicted_original = np.round(predicted_original, 2)

    mse_score = mse(actual_original.flatten(), predicted_original.flatten())
    rmse_score = sqrt(mse_score)
    mae_score = mae(actual_original.flatten(), predicted_original.flatten())
    mre_score = mre(actual_original.flatten(), predicted_original.flatten())
    final_r2_score = r2_score(actual_original.flatten(), predicted_original.flatten())

    return mse_score, rmse_score, mae_score, mre_score, final_r2_score, predicted_original


def summarize_scores(name,
                     mse_score,
                     rmse_score,
                     mae_score,
                     mre_score,
                     final_r2_score,
                     train_time=None,
                     inference_time=None,
                     company=None):

    print(f"\n************* {name} *************")
    print("MAE :", mae_score)
    print("MRE :", mre_score)
    print("MSE :", mse_score)
    print("RMSE:", rmse_score)
    print("R2  :", final_r2_score)

    if train_time is not None:
        print("Training Time (s):", round(train_time, 4))
    if inference_time is not None:
        print("Inference Time (s):", round(inference_time, 4))

    all_res.append({
        "Company": company,
        "Model": name,
        "MAE": mae_score,
        "MRE": mre_score,
        "MSE": mse_score,
        "RMSE": rmse_score,
        "R2": final_r2_score,
        "Train_Time": 0 if train_time is None else train_time,
        "Inference_Time": 0 if inference_time is None else inference_time
    })


# =========================================================
# RANDOM WALK WITH DRIFT
# =========================================================
def random_walk_with_drift(input_sequence, forecast_horizon=5):
    input_sequence = np.array(input_sequence)
    drift = np.mean(np.diff(input_sequence))
    last_value = input_sequence[-1]
    forecast = [last_value + drift * (i + 1) for i in range(forecast_horizon)]
    return np.array(forecast)


def random_walk_predict_from_test(test_original_series, n_input=5, horizon=5):
    """
    test_original_series: original close prices from test partition only
    Returns predictions in ORIGINAL SCALE
    """
    values = test_original_series.values
    preds = []

    for i in range(len(values) - n_input - horizon + 1):
        seq = values[i:i+n_input]
        forecast = random_walk_with_drift(seq, horizon)
        preds.append(forecast)

    return np.array(preds)


# =========================================================
# ARIMA BASELINE
# =========================================================
def arima_forecast(train_original_series, n_test_samples, horizon=5, order=(5,1,0)):
    """
    Trains once on original training close prices.
    Produces same multi-step forecast for each rolling evaluation point.
    This is a simple baseline, not rolling re-fit.
    Returns predictions in ORIGINAL SCALE.
    """
    start_train = time.perf_counter()

    model = ARIMA(train_original_series, order=order)
    model_fit = model.fit()

    train_time = time.perf_counter() - start_train

    start_inf = time.perf_counter()

    preds = []
    for _ in range(n_test_samples):
        forecast = model_fit.forecast(steps=horizon)
        forecast = np.array(forecast).flatten()

        if len(forecast) < horizon:
            forecast = np.pad(forecast, (0, horizon - len(forecast)), mode='edge')

        preds.append(forecast)

    preds = np.vstack(preds)

    inference_time = time.perf_counter() - start_inf

    return preds, train_time, inference_time


# =========================================================
# XGBOOST BASELINE
# =========================================================
def build_xgboost_model():
    return XGBRegressor(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective='reg:squarederror',
        random_state=42
    )


def train_xgboost(train_x, train_y):
    """
    train_x: (samples, window, features)
    train_y: (samples, horizon, 1) usually squeezed to (samples, horizon)
    We train direct multi-output by fitting one model per horizon step.
    """

    X_train = train_x.reshape(train_x.shape[0], -1)
    y_train = np.squeeze(train_y)

    if y_train.ndim == 1:
        y_train = y_train.reshape(-1, 1)

    models = []

    start_train = time.perf_counter()

    for h in range(y_train.shape[1]):
        model = build_xgboost_model()
        model.fit(X_train, y_train[:, h])
        models.append(model)

    train_time = time.perf_counter() - start_train

    return models, train_time


def predict_xgboost(models, test_x):
    """
    Direct multi-output prediction using one model per horizon step.
    Returns predictions in SCALED space because XGBoost trained on scaled y.
    """
    X_test = test_x.reshape(test_x.shape[0], -1)

    start_inf = time.perf_counter()

    preds = []
    for model in models:
        preds.append(model.predict(X_test))

    preds = np.stack(preds, axis=1)

    inference_time = time.perf_counter() - start_inf

    return preds, inference_time



# =========================================================
# BASE MODELS
# =========================================================
def build_birnn_model(input_shape, horizon=5, units=128):
    model = Sequential()
    model.add(Bidirectional(SimpleRNN(units, activation='relu'), input_shape=input_shape))
    model.add(Dense(horizon))
    model.compile(loss='mse', optimizer='adam')
    return model


def build_gru_model(input_shape, horizon=5, units=128):
    model = Sequential()
    model.add(GRU(units, activation='relu', input_shape=input_shape))
    model.add(Dense(horizon))
    model.compile(loss='mse', optimizer='adam')
    return model


def build_lstm_model(input_shape, horizon=5, units=128):
    model = Sequential()
    model.add(LSTM(units, activation='relu', input_shape=input_shape))
    model.add(Dense(horizon))
    model.compile(loss='mse', optimizer='adam')
    return model

def evaluate_model_with_timing(test_x, test_y, model):
    start_inf = time.perf_counter()
    predictions = model.predict(test_x, verbose=0)
    inference_time = time.perf_counter() - start_inf

    predictions = array(predictions)
    mse_score, rmse_score, mae_score, mre_score,  final_r2_score, y_hat = evaluate_forecasts(test_y, predictions)

    return mse_score, rmse_score, mae_score, mre_score, final_r2_score, y_hat, inference_time


def train_and_eval_dl_model(model_name,
                            model,
                            train_x,
                            train_y,
                            val_x,
                            val_y,
                            test_x,
                            test_y_eval,
                            company_name,
                            save_dir=None):

    verbose, epochs, batch_size = 0, 32, max(1, math.floor(len(train_x) * 0.05))

    start_train = time.perf_counter()
    model.fit(
        train_x,
        train_y,
        epochs=epochs,
        batch_size=batch_size,
        verbose=verbose,
        validation_data=(val_x, val_y)
    )
    train_time = time.perf_counter() - start_train

    mse_score, rmse_score, mae_score, mre_score, r2, preds, inference_time = evaluate_model_with_timing(
        test_x,
        test_y_eval,
        model
    )

    summarize_scores(
        model_name,
        mse_score,
        rmse_score,
        mae_score,
        mre_score,
        r2,
        train_time,
        inference_time,
        company_name
    )

    # if save_dir is not None:
    #     os.makedirs(save_dir, exist_ok=True)
    #     with open(os.path.join(save_dir, f"{company_name}_{model_name}.pkl"), "wb") as f:
    #         pickle.dump(model, f)

    return model, preds


# =========================================================
# MODEL LOADING
# =========================================================
def load_all_models(company_name, dirname):
    all_models = []
    model_order = ["BRNN", "GRU", "LSTM"]

    for filename in os.listdir(dirname):
        root, ext = os.path.splitext(filename)
        if root.startswith(str(company_name)) and ext == '.pkl':
            with open(os.path.join(dirname, filename), 'rb') as file:
                model = pickle.load(file)
                all_models.append((root, model))

    all_models_sorted = sorted(
        all_models,
        key=lambda x: model_order.index(x[0].split('_')[-1]) if x[0].split('_')[-1] in model_order else model_order.index(x[0].split('_')[-2])
    )

    print("**********Models Order***************")
    [print(x[0]) for x in all_models_sorted]
    print("*************************************")

    all_models_sorted = [model for _, model in all_models_sorted]
    return all_models_sorted


# =========================================================
# ENSEMBLES
# =========================================================
def stacked_dataset(members, inputX):
    stackX = None
    for model in members:
        yhat = model.predict(inputX, verbose=0)
        if stackX is None:
            stackX = yhat
        else:
            stackX = np.dstack((stackX, yhat))
    return stackX


def averaging_ensemble(company_name, members, test_x, test_y_eval,models_name = ""):

    models_used = []
    if(models_name == "B_G"):
      models_used = [members[0], members[1]]
    elif(models_name == "B_L"):
      models_used = [members[0], members[2]]
    elif(models_name == "G_L"):
      models_used = [members[1], members[2]]
    else:
      models_used = members


    stacked_test = stacked_dataset(models_used, test_x)

    start_inf = time.perf_counter()
    avg_preds = np.mean(stacked_test, axis=2)
    inference_time = time.perf_counter() - start_inf

    mse_score, rmse_score, mae_score,mre_score, r2, preds = evaluate_forecasts(test_y_eval, avg_preds)

    summarize_scores(
        "Averaging Ensemble",
        mse_score,
        rmse_score,
        mae_score,
        mre_score,
        r2,
        0.0,
        inference_time,
        company_name
    )
    return preds


# =========================================================
# LEAKAGE-SAFE STACKING
# Train meta-learner on VALIDATION predictions only
# =========================================================
def build_meta_ann(input_shape):
    model = Sequential()
    model.add(Flatten(input_shape=input_shape))
    model.add(Dense(128, activation='relu'))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(5))
    model.compile(optimizer='adam', loss='mse')
    return model


def build_meta_gru(input_shape):
    model = Sequential()
    model.add(GRU(128, activation='relu', input_shape=input_shape))
    model.add(Dense(5))
    model.compile(optimizer='adam', loss='mse')
    return model


def build_meta_lstm(input_shape):
    model = Sequential()
    model.add(LSTM(128, activation='relu', input_shape=input_shape))
    model.add(Dense(5))
    model.compile(optimizer='adam', loss='mse')
    return model


def train_eval_stacking_safe(company_name,
                             members,
                             train_x,
                             train_y,
                             val_x,
                             val_y,
                             test_x,
                             test_y_eval,
                             meta_type="ANN",
                             models_name = ""):
    models_used = []
    if(models_name == "B_G"):
      models_used = [members[0], members[1]]
    elif(models_name == "B_L"):
      models_used = [members[0], members[2]]
    elif(models_name == "G_L"):
      models_used = [members[1], members[2]]
    else:
      models_used = members

    # IMPORTANT:
    # Validation predictions are used to train the meta-learner.
    stacked_val = stacked_dataset(models_used, val_x)
    stacked_test = stacked_dataset(models_used, test_x)

    if meta_type == "ANN":
        model = build_meta_ann((stacked_val.shape[1], stacked_val.shape[2]))
    elif meta_type == "GRU":
        model = build_meta_gru((stacked_val.shape[1], stacked_val.shape[2]))
    elif meta_type == "LSTM":
        model = build_meta_lstm((stacked_val.shape[1], stacked_val.shape[2]))
    else:
        raise ValueError("meta_type must be ANN, GRU, or LSTM")

    start_train = time.perf_counter()
    model.fit(
        stacked_val,
        val_y,
        epochs=100,
        verbose=0
    )
    train_time = time.perf_counter() - start_train

    start_inf = time.perf_counter()
    preds = model.predict(stacked_test, verbose=0)
    inference_time = time.perf_counter() - start_inf

    mse_score, rmse_score, mae_score, mre_score, r2, preds_denorm = evaluate_forecasts(test_y_eval, preds)

    summarize_scores(
      name=f"Stacking {meta_type} {models_name}",
      mse_score=mse_score,
      rmse_score=rmse_score,
      mae_score=mae_score,
      mre_score=mre_score,
      final_r2_score=r2,
      train_time=train_time,
      inference_time=inference_time,
      company=company_name
    )

    return preds_denorm


# =========================================================
# MAIN PIPELINE FOR ONE COMPANY
# =========================================================
def run_company_pipeline(selected_companies_dic,
                         company_name,
                         best_models_dir):


    company = pd.read_csv(f'/content/drive/MyDrive/KFUPM/Graduation/data/Saudi/{company_name}_selected_features.csv',index_col='Unnamed: 0',parse_dates=True)
    train_x, train_y, val_x, val_y, test_x, test_y, test = \
        prepare_company_for_training_and_testing(company, selected_ti)

    test_y_eval = true_y_to_evaluation_form(test.iloc[:, 3], 5, 5)

    # -------- Train base DL models --------
    input_shape = (train_x.shape[1], train_x.shape[2])

    birnn_model = build_birnn_model(input_shape, units=selected_companies_dic[company_name][2][0] )
    gru_model = build_gru_model(input_shape, units=selected_companies_dic[company_name][2][1] )
    lstm_model = build_lstm_model(input_shape, units=selected_companies_dic[company_name][2][2] )

    birnn_model, birnn_preds = train_and_eval_dl_model(
        "BRNN", birnn_model, train_x, train_y, val_x, val_y, test_x, test_y_eval, company_name, best_models_dir
    )
    gru_model, gru_preds = train_and_eval_dl_model(
        "GRU", gru_model, train_x, train_y, val_x, val_y, test_x, test_y_eval, company_name, best_models_dir
    )
    lstm_model, lstm_preds = train_and_eval_dl_model(
        "LSTM", lstm_model, train_x, train_y, val_x, val_y, test_x, test_y_eval, company_name, best_models_dir
    )

    members = [birnn_model, gru_model, lstm_model]

    # -------- Baselines --------
    selected_ti_B = list(company.columns)
    if 'close' in selected_ti:
        selected_ti_B.remove('close')

    train_x_b, train_y_b, val_x_b, val_y_b, test_x_b, test_y_b, train_df_b, val_df_b, test_df_b = \
        prepare_company_for_training_and_testing_base(company, selected_ti_B)

    # actual values for evaluation are the scaled test_y from pipeline
    # evaluation function will inverse-transform them internally

    # =====================================================
    # RANDOM WALK
    # =====================================================
    print(f"\nRunning Random Walk for {company_name}")

    rw_start_inf = time.perf_counter()

    rw_preds_original = random_walk_predict_from_test(
        test_original_series=test_df_b['close'],
        n_input=5,
        horizon=5
    )

    rw_inf_time = time.perf_counter() - rw_start_inf

    rw_mse, rw_rmse, rw_mae, rw_mre, rw_r2, rw_preds_eval = evaluate_forecasts(
        actual=test_y_b,
        predicted_scaled_or_original=rw_preds_original,
        actual_are_scaled=True,
        predictions_are_scaled=False
    )

    summarize_scores(
        "Random Walk with Drift",
        rw_mse,
        rw_rmse,
        rw_mae,
        rw_mre,
        rw_r2,
        train_time=0.0,
        inference_time=rw_inf_time,
        company=company_name
    )

    # =====================================================
    # ARIMA
    # =====================================================
    print(f"\nRunning ARIMA for {company_name}")

    arima_preds_original, arima_train_time, arima_inf_time = arima_forecast(
        train_original_series=train_df_b['close'],
        n_test_samples=len(test_y),
        horizon=5,
        order=(5,1,0)
    )

    arima_mse, arima_rmse, arima_mae, arima_mre, arima_r2, arima_preds_eval = evaluate_forecasts(
        actual=test_y_b,
        predicted_scaled_or_original=arima_preds_original,
        actual_are_scaled=True,
        predictions_are_scaled=False
    )

    summarize_scores(
        "ARIMA",
        arima_mse,
        arima_rmse,
        arima_mae,
        arima_mre,
        arima_r2,
        train_time=arima_train_time,
        inference_time=arima_inf_time,
        company=company_name
    )

    # =====================================================
    # XGBOOST
    # =====================================================
    print(f"\nRunning XGBoost for {company_name}")

    xgb_models, xgb_train_time = train_xgboost(train_x_b, train_y_b)

    xgb_preds_scaled, xgb_inf_time = predict_xgboost(xgb_models, test_x_b)

    xgb_mse, xgb_rmse, xgb_mae, xgb_mre, xgb_r2, xgb_preds_eval = evaluate_forecasts(
        actual=test_y_b,
        predicted_scaled_or_original=xgb_preds_scaled,
        actual_are_scaled=True,
        predictions_are_scaled=True
    )

    summarize_scores(
        "XGBoost",
        xgb_mse,
        xgb_rmse,
        xgb_mae,
        xgb_mre,
        xgb_r2,
        train_time=xgb_train_time,
        inference_time=xgb_inf_time,
        company=company_name
    )

    # -------- Ensembles --------
    avg_preds = averaging_ensemble(company_name, members, test_x, test_y_eval)
    avg_B_G_preds = averaging_ensemble(company_name, members, test_x, test_y_eval,"B_G")
    avg_B_L_preds = averaging_ensemble(company_name, members, test_x, test_y_eval,"B_L")
    avg_G_L_preds = averaging_ensemble(company_name, members, test_x, test_y_eval,"G_L")
    stk_ann_preds = train_eval_stacking_safe(company_name, members, train_x, train_y, val_x, val_y, test_x, test_y_eval, "ANN")
    stk_ann_BG_preds = train_eval_stacking_safe(company_name, members, train_x, train_y, val_x, val_y, test_x, test_y_eval, "ANN","B_G")
    stk_ann_BL_preds = train_eval_stacking_safe(company_name, members, train_x, train_y, val_x, val_y, test_x, test_y_eval, "ANN","B_L")
    stk_ann_GL_preds = train_eval_stacking_safe(company_name, members, train_x, train_y, val_x, val_y, test_x, test_y_eval, "ANN","G_L")
    stk_gru_preds = train_eval_stacking_safe(company_name, members, train_x, train_y, val_x, val_y, test_x, test_y_eval, "GRU")
    stk_gru_BG_preds = train_eval_stacking_safe(company_name, members, train_x, train_y, val_x, val_y, test_x, test_y_eval, "GRU","B_G")
    stk_gru_BL_preds = train_eval_stacking_safe(company_name, members, train_x, train_y, val_x, val_y, test_x, test_y_eval, "GRU","B_L")
    stk_gru_GL_preds = train_eval_stacking_safe(company_name, members, train_x, train_y, val_x, val_y, test_x, test_y_eval, "GRU","G_L")
    stk_lstm_preds = train_eval_stacking_safe(company_name, members, train_x, train_y, val_x, val_y, test_x, test_y_eval, "LSTM")
    stk_lstm_BG_preds = train_eval_stacking_safe(company_name, members, train_x, train_y, val_x, val_y, test_x, test_y_eval, "LSTM","B_G")
    stk_lstm_BL_preds = train_eval_stacking_safe(company_name, members, train_x, train_y, val_x, val_y, test_x, test_y_eval, "LSTM","B_L")
    stk_lstm_GL_preds = train_eval_stacking_safe(company_name, members, train_x, train_y, val_x, val_y, test_x, test_y_eval, "LSTM","G_L")


    models_predictions = {
        "RandomWalk": rw_preds_eval,
        "ARIMA": arima_preds_eval,
        "XGBoost": xgb_preds_eval,
        "BRNN": birnn_preds,
        "GRU": gru_preds,
        "LSTM": lstm_preds,
        "Avg": avg_preds,
        "Avg_B_G": avg_B_G_preds,
        "Avg_B_L": avg_B_L_preds,
        "Avg_G_L": avg_G_L_preds,
        "STK_ANN": stk_ann_preds,
        "STK_ANN_BG": stk_ann_BG_preds,
        "STK_ANN_BL": stk_ann_BL_preds,
        "STK_ANN_GL": stk_ann_GL_preds,
        "STK_GRU": stk_gru_preds,
        "STK_GRU_BG": stk_gru_BG_preds,
        "STK_GRU_BL": stk_gru_BL_preds,
        "STK_GRU_GL": stk_gru_GL_preds,
        "STK_LSTM": stk_lstm_preds,
        "STK_LSTM_BG": stk_lstm_BG_preds,
        "STK_LSTM_BL": stk_lstm_BL_preds,
        "STK_LSTM_GL": stk_lstm_GL_preds
    }

    return models_predictions



# =========================================================
# COMPUTATIONAL COST EXPORT
# =========================================================
def export_computational_cost_tables():
    df = pd.DataFrame(time_results)

    avg_df = df.groupby("Model")[["Train_Time", "Inference_Time"]].mean()

    latex_table = avg_df.to_latex(
        float_format="%.4f",
        caption="Average training and inference time across the ten Saudi companies.",
        label="tab:computational_cost"
    )

    print(latex_table)
    return df, avg_df, latex_table

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

# =========================================================
# RMSE PER BLOCK
# =========================================================
def compute_rmse_blocks(actual, predicted, horizon=5):

    actual = np.array(actual).reshape(-1)
    predicted = np.array(predicted).reshape(-1)

    rmse_scores = []

    for i in range(0, len(actual), horizon):

        if i + horizon > len(actual):
            break

        a = actual[i:i+horizon]
        p = predicted[i:i+horizon]

        rmse = np.sqrt(np.mean((a - p) ** 2))

        rmse_scores.append(rmse)

    return np.array(rmse_scores)


# =========================================================
# CLIFF'S DELTA (EFFECT SIZE)
# =========================================================
def cliffs_delta(x, y):

    greater = 0
    smaller = 0

    for xi in x:
        for yi in y:
            if xi > yi:
                greater += 1
            elif xi < yi:
                smaller += 1

    delta = (greater - smaller) / (len(x) * len(y))

    return delta


# =========================================================
# WILCOXON TEST WITH BONFERRONI
# =========================================================
def wilcoxon_test(actual, pred1, pred2, comparisons):

    rmse1 = compute_rmse_blocks(actual, pred1)
    rmse2 = compute_rmse_blocks(actual, pred2)

    stat, p = wilcoxon(rmse1, rmse2)

    # Bonferroni correction
    p_corrected = min(p * comparisons, 1.0)

    effect = cliffs_delta(rmse1, rmse2)

    return stat, p_corrected, effect, rmse1.mean(), rmse2.mean()


# =========================================================
# BUILD MATRICES
# =========================================================
def construct_wilcoxon_matrix(models_predictions, actual):

    model_names = list(models_predictions.keys())

    n = len(model_names)

    comparisons = n * (n - 1) / 2

    wx = pd.DataFrame(0, index=model_names, columns=model_names)
    pvals = pd.DataFrame(0.0, index=model_names, columns=model_names)
    effects = pd.DataFrame(0.0, index=model_names, columns=model_names)

    for i in range(n):
        for j in range(i + 1, n):

            m1 = model_names[i]
            m2 = model_names[j]

            stat, p, effect, err1, err2 = wilcoxon_test(
                actual,
                models_predictions[m1],
                models_predictions[m2],
                comparisons
            )

            pvals.loc[m1, m2] = p
            effects.loc[m1, m2] = effect

            if p < 0.05:

                if err1 < err2:
                    wx.loc[m1, m2] = 1
                else:
                    wx.loc[m2, m1] = 1

    return wx, pvals, effects


# =========================================================
# MODEL RANKING
# =========================================================
def summarize_wins_losses(wx_matrix):

    wins = wx_matrix.sum(axis=1)
    losses = wx_matrix.sum(axis=0)

    summary = pd.DataFrame({
        "Wins": wins,
        "Losses": losses
    })

    summary["Score"] = wins - losses
    summary = summary.sort_values("Score", ascending=False)

    return summary


# =========================================================
# LATEX EXPORT
# =========================================================
def export_latex_tables(wx, summary):

    latex_matrix = wx.to_latex(
        caption="Wilcoxon signed-rank test win matrix.",
        label="tab:wilcoxon_matrix"
    )

    latex_summary = summary.to_latex(
        caption="Model ranking based on Wilcoxon wins and losses.",
        label="tab:model_ranking"
    )

    print("\nLATEX TABLE - WILCOXON MATRIX\n")
    print(latex_matrix)

    print("\nLATEX TABLE - MODEL RANKING\n")
    print(latex_summary)


# =========================================================
# MAIN FUNCTION
# =========================================================
def run_wilcoxon_analysis(actual, models_predictions):

    print("Running Wilcoxon statistical comparison...\n")

    wx, pvals, effects = construct_wilcoxon_matrix(
        models_predictions,
        actual
    )

    ranking = summarize_wins_losses(wx)

    print("\n===== WILCOXON WIN MATRIX =====\n")
    print(wx)

    print("\n===== MODEL RANKING =====\n")
    print(ranking)

    print("\n===== P-VALUES =====\n")
    print(pvals)

    print("\n===== EFFECT SIZE (CLIFF DELTA) =====\n")
    print(effects)

    export_latex_tables(wx, ranking)

    return wx, ranking, pvals, effects

In [ ]:
selected_companies_dic = {"ADC":["ADC","2001-12-31",[200,256,500]], "ALRAJHI":["ALRAJHI","2001-12-31",[900,800,400]], "GACO":["GACO","2002-01-01",[900,700,1000]], "EMAAR EC":["EMAAR EC","2006-10-06",[1024,600,1000]], "FITAIHI GROUP":["FITAIHI GROUP","2002-01-26",[200,512,900]], "SARCO":["SARCO","2002-01-23",[500,800,1024]], "SAUDI ELECTRICITY":["SAUDI ELECTRICITY","2002-06-08",[300,243,1024]], "SPIMACO":["SPIMACO","2001-12-31",[900,256,600]], "STC":["STC","2003-01-25",[1024,729,1024]], "TASNEE":["TASNEE","2001-12-31",[1024,512,1024]] }

In [ ]:
selected_ti = [
  'open','high','low','close','alma','decay','dema','ema','fwma','hma',
  'hl2','hlc3','hwma','jma','kama','linreg','median','midpoint','midprice',
  'pwma','quantile','rma','sinwma','sma','ssf','swma','t3','tema','trima',
  'vwap','vmwa','wcp','wma','zlma'
]

In [ ]:
all_preds = []
all_res = []

for company_name in selected_companies_dic.keys():
    result = run_company_pipeline(
        selected_companies_dic=selected_companies_dic,
        company_name=company_name,
        best_models_dir=f"/content/drive/MyDrive/KFUPM/Graduation/final_experiments/best_models/{company_name}"
    )
    all_preds.append(result)

results_df = pd.DataFrame(all_res)

results_df.to_csv(
    "/content/drive/MyDrive/KFUPM/Graduation/final_experiments/all_models_results.csv",
    index=False
)

print("Results saved successfully.")

Train shape: (5114, 34), Val shape: (366, 34), Test shape: (1160, 34)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



************* BRNN *************
MAE : 0.32320591053767994
MRE : 0.028533960357577503
MSE : 0.15722333289120313
RMSE: 0.39651397565685265
R2  : 0.9260119235736975
Training Time (s): 24.1959
Inference Time (s): 0.6496

************* GRU *************
MAE : 0.19990964793081806
MRE : 0.017208304047794217
MSE : 0.08227708044144336
RMSE: 0.28683981669469
R2  : 0.9612810464967878
Training Time (s): 52.7157
Inference Time (s): 0.9371

************* LSTM *************
MAE : 0.19023979558870135
MRE : 0.016301390622054417
MSE : 0.07748022643202516
RMSE: 0.27835270149941993
R2  : 0.9635384086486272
Training Time (s): 169.0135
Inference Time (s): 1.2517

Running Random Walk for ADC

************* Random Walk with Drift *************
MAE : 0.22774456993918332
MRE : 0.019303197132232615
MSE : 0.1271900086880973
RMSE: 0.35663708260372656
R2  : 0.9401453721249567
Training Time (s): 0.0
Inference Time (s): 0.0268

Running ARIMA for ADC


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)



************* ARIMA *************
MAE : 1.4814248479582972
MRE : 0.13354093485661234
MSE : 2.9989923197219808
RMSE: 1.7317598908976906
R2  : -0.4113024375779468
Training Time (s): 1.0324
Inference Time (s): 3.3509

Running XGBoost for ADC

************* XGBoost *************
MAE : 0.18803824270134073
MRE : 0.016067182511429027
MSE : 0.07862200066952785
RMSE: 0.2803961495269289
R2  : 0.9630010985815242
Training Time (s): 17.2923
Inference Time (s): 0.03

************* Averaging Ensemble *************
MAE : 0.21555343526513757
MRE : 0.0186645611709241
MSE : 0.08698180520470747
RMSE: 0.29492677939567896
R2  : 0.9590670396509339
Training Time (s): 0.0
Inference Time (s): 0.0002

************* Averaging Ensemble *************
MAE : 0.24894527083992649
MRE : 0.021729917154505347
MSE : 0.1043664622622971
RMSE: 0.32305798591320584
R2  : 0.9508859553845669
Training Time (s): 0.0
Inference Time (s): 0.0002

************* Averaging Ensemble *************
MAE : 0.2284396225757748
MRE : 0.01985337

...


************* Stacking LSTM B_G *************
MAE : 0.5043197208672996
MRE : 0.02849177005704338
MSE : 0.5265558024413567
RMSE: 0.7256416487780706
R2  : 0.9275294783432101
Training Time (s): 16.2655
Inference Time (s): 0.4636


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



************* Stacking LSTM B_L *************
MAE : 0.46135012875713544
MRE : 0.026944610826242808
MSE : 0.4113714253125513
RMSE: 0.6413824329622314
R2  : 0.9433824456042947
Training Time (s): 16.5089
Inference Time (s): 0.4979


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



************* Stacking LSTM G_L *************
MAE : 0.5059183298039706
MRE : 0.028670454597088748
MSE : 0.5236553310556034
RMSE: 0.7236403326622994
R2  : 0.9279286737815693
Training Time (s): 16.0886
Inference Time (s): 0.4845
Results saved successfully.


In [24]:
baseline_models = ["RandomWalk", "ARIMA", "XGBoost"]

for i, company_name in enumerate(selected_companies_dic):

    preds = all_preds[i]

    df_list = []

    for model in baseline_models:

        pred = preds[model]

        pred_df = pd.DataFrame(
            pred,
            columns=[f"{model}_t+{j+1}" for j in range(pred.shape[1])]
        )

        df_list.append(pred_df)

    baseline_predictions = pd.concat(df_list, axis=1)

    baseline_predictions.to_csv(
        f"/content/drive/MyDrive/KFUPM/Graduation/final_experiments/predictions/{company_name}_baseline_predictions.csv",
        index=False
    )

print("Baseline predictions saved successfully.")

Baseline predictions saved successfully.


In [22]:
all_preds[0]['RandomWalk']

array([14.23, 14.62, 15.01, 15.4 , 15.79])

In [25]:
import pandas as pd
import numpy as np

def convert_all_models_to_block_format(df, horizon=5):

    # أسماء النماذج (كل الأعمدة ماعدا Actual)
    models = [c for c in df.columns if c != "Actual"]

    blocks = []

    for start in range(0, len(df), horizon):

        if start + horizon <= len(df):

            row = {}

            for model in models:

                values = df[model].iloc[start:start+horizon].values

                for step in range(horizon):

                    row[f"{model}_t+{step+1}"] = values[step]

            # actual values
            actual_vals = df["Actual"].iloc[start:start+horizon].values

            for step in range(horizon):
                row[f"Actual_t+{step+1}"] = actual_vals[step]

            blocks.append(row)

    df_blocks = pd.DataFrame(blocks)

    return df_blocks

In [26]:
import os
import pandas as pd

input_path = "/content/drive/MyDrive/KFUPM/Graduation/final_experiments/results/Saudi/Forecasts"
output_path = "/content/drive/MyDrive/KFUPM/Graduation/final_experiments/predictions"

os.makedirs(output_path, exist_ok=True)

for file in os.listdir(input_path):

    if file.endswith("_forcasts_all.csv"):

        company_name = file.replace("_forcasts_all.csv", "")

        file_path = os.path.join(input_path, file)

        df = pd.read_csv(file_path)

        df_blocks = convert_all_models_to_block_format(df)

        save_path = os.path.join(
            output_path,
            f"{company_name}_block_predictions.csv"
        )

        df_blocks.to_csv(save_path, index=False)

        print(f"{company_name} converted successfully")

ADC converted successfully
ALRAJHI converted successfully
STC converted successfully
GACO converted successfully
SPIMACO converted successfully
SAUDI ELECTRICITY converted successfully
FITAIHI GROUP converted successfully
EMAAR EC converted successfully
TASNEE converted successfully
SARCO converted successfully


In [31]:
import pandas as pd
from pathlib import Path

# folder containing files
folder = Path("/content/drive/MyDrive/KFUPM/Graduation/final_experiments/predictions")

for block_file in folder.glob("*block_prediction*.csv"):

    company = block_file.name.split("_block")[0]
    baseline_file = folder / f"{company}_baseline_predictions.csv"

    print(f"\nProcessing {company}")

    if not baseline_file.exists():
        print("Baseline file missing")
        continue

    # read files
    block_df = pd.read_csv(block_file)
    baseline_df = pd.read_csv(baseline_file)

    # remove first row from baseline
    baseline_df = baseline_df.iloc[1:].reset_index(drop=True)

    # find Random Walk column automatically
    rw_col = [c for c in baseline_df.columns if "Random" in c][0]

    # make sure lengths match
    baseline_df = baseline_df.iloc[:len(block_df)]

    # insert Random Walk into block
    block_df.insert(0, "Baseline_Random_Walk", baseline_df[rw_col].values)

    # save merged file
    output_file = folder / f"{company}_merged_predictions.csv"
    block_df.to_csv(output_file, index=False)

    print("Saved:", output_file)


Processing ADC
Saved: /content/drive/MyDrive/KFUPM/Graduation/final_experiments/predictions/ADC_merged_predictions.csv

Processing ALRAJHI
Saved: /content/drive/MyDrive/KFUPM/Graduation/final_experiments/predictions/ALRAJHI_merged_predictions.csv

Processing STC
Saved: /content/drive/MyDrive/KFUPM/Graduation/final_experiments/predictions/STC_merged_predictions.csv

Processing GACO
Saved: /content/drive/MyDrive/KFUPM/Graduation/final_experiments/predictions/GACO_merged_predictions.csv

Processing SPIMACO
Saved: /content/drive/MyDrive/KFUPM/Graduation/final_experiments/predictions/SPIMACO_merged_predictions.csv

Processing SAUDI ELECTRICITY
Saved: /content/drive/MyDrive/KFUPM/Graduation/final_experiments/predictions/SAUDI ELECTRICITY_merged_predictions.csv

Processing FITAIHI GROUP
Saved: /content/drive/MyDrive/KFUPM/Graduation/final_experiments/predictions/FITAIHI GROUP_merged_predictions.csv

Processing EMAAR EC
Saved: /content/drive/MyDrive/KFUPM/Graduation/final_experiments/predicti

In [40]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

folder = Path("/content/drive/MyDrive/KFUPM/Graduation/final_experiments/predictions")

WINDOW = 5
alpha = 0.05

pairwise_results = []
win_loss = {}

for file in folder.glob("*merged_predictions.csv"):

    print("Processing:", file.name)

    company = file.stem.split("_")[0]

    df = pd.read_csv(file)
    df.columns = df.columns.str.strip()

    actual_cols = [c for c in df.columns if "actual" in c.lower()]
    models = [c for c in df.columns if c not in actual_cols]

    for actual_col in actual_cols:

        horizon = actual_col.replace("Actual_", "")

        # -----------------------------
        # compute RMSE windows
        # -----------------------------
        rmse_windows = {}

        for model in models:

            rmse_list = []

            for k in range(0, len(df), WINDOW):

                y_true = df[actual_col].iloc[k:k+WINDOW].values
                y_pred = df[model].iloc[k:k+WINDOW].values

                if len(y_true) == WINDOW:

                    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
                    rmse_list.append(rmse)

            rmse_windows[model] = np.array(rmse_list)

        # -----------------------------
        # pairwise comparisons
        # -----------------------------
        for i in range(len(models)):
            for j in range(i+1, len(models)):

                m1 = models[i]
                m2 = models[j]

                rmse1 = rmse_windows[m1]
                rmse2 = rmse_windows[m2]

                stat, p = wilcoxon(rmse1, rmse2)

                pairwise_results.append({
                    "Company": company,
                    "Horizon": horizon,
                    "Model1": m1,
                    "Model2": m2,
                    "Statistic": stat,
                    "p_value": p,
                    "mean_rmse_m1": rmse1.mean(),
                    "mean_rmse_m2": rmse2.mean()
                })

results_df = pd.DataFrame(pairwise_results)

# -----------------------------
# Holm correction
# -----------------------------
reject, p_corr, _, _ = multipletests(
    results_df["p_value"],
    method="holm"
)

results_df["p_corrected"] = p_corr
results_df["Significant"] = reject

# -----------------------------
# compute wins/losses
# -----------------------------
models = set(results_df["Model1"]).union(set(results_df["Model2"]))

for m in models:
    win_loss[m] = {"wins":0,"losses":0,"ties":0}

for _, row in results_df.iterrows():

    m1 = row["Model1"]
    m2 = row["Model2"]

    if not row["Significant"]:
        win_loss[m1]["ties"] += 1
        win_loss[m2]["ties"] += 1
        continue

    if row["mean_rmse_m1"] < row["mean_rmse_m2"]:
        win_loss[m1]["wins"] += 1
        win_loss[m2]["losses"] += 1
    else:
        win_loss[m2]["wins"] += 1
        win_loss[m1]["losses"] += 1

# -----------------------------
# summary table
# -----------------------------
summary = []

for m,v in win_loss.items():

    total = v["wins"] + v["losses"]

    summary.append({
        "Model": m,
        "Wins": v["wins"],
        "Losses": v["losses"],
        "WinRate_%": round(v["wins"]/total*100,2) if total>0 else 0
    })

summary_df = pd.DataFrame(summary).sort_values("Wins",ascending=False)

results_df.to_csv("wilcoxon_pairwise_results_corrected.csv",index=False)
summary_df.to_csv("wilcoxon_model_ranking_corrected.csv",index=False)

print(summary_df)

Processing: ADC_merged_predictions.csv
Processing: ALRAJHI_merged_predictions.csv


/usr/local/lib/python3.12/dist-packages/scipy/stats/_wilcoxon.py:178: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se


Processing: STC_merged_predictions.csv


/usr/local/lib/python3.12/dist-packages/scipy/stats/_wilcoxon.py:178: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se


Processing: GACO_merged_predictions.csv


...

Processing: FITAIHI GROUP_merged_predictions.csv


/usr/local/lib/python3.12/dist-packages/scipy/stats/_wilcoxon.py:178: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se


Processing: EMAAR EC_merged_predictions.csv
Processing: TASNEE_merged_predictions.csv


/usr/local/lib/python3.12/dist-packages/scipy/stats/_wilcoxon.py:178: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se


Processing: SARCO_merged_predictions.csv
                        Model  Wins  Losses  WinRate_%
6                 XGBoost_t+1  2900     440      86.83
9    Baseline_Random Walk_t+1  2619    2011      56.57
25   Baseline_Random Walk_t+2  2004    2784      41.85
89                XGBoost_t+2  1579     555      73.99
103  Baseline_Random Walk_t+3  1463    3165      31.61
..                        ...   ...     ...        ...
36                  ARIMA_t+2   110    5565       1.94
42                  ARIMA_t+1   105    5575       1.85
4                   ARIMA_t+4    80    5595       1.41
43                  ARIMA_t+3    80    5580       1.41
99                  ARIMA_t+5    55    5615       0.97

[115 rows x 4 columns]


In [46]:
import pandas as pd

df = pd.read_csv("wilcoxon_model_ranking_corrected.csv")

# remove horizon suffix
df["BaseModel"] = df["Model"].str.replace(r"_t\+\d+", "", regex=True)

# aggregate wins and losses
agg = (
    df.groupby("BaseModel")
    .agg({
        "Wins": "sum",
        "Losses": "sum"
    })
    .reset_index()
)

# compute win rate
agg["WinRate_%"] = round(
    agg["Wins"] / (agg["Wins"] + agg["Losses"]) * 100,
    2
)

# sort by wins
agg = agg.sort_values("Wins", ascending=False)

print(agg)

agg.to_csv("wilcoxon_model_ranking_aggregated.csv", index=False)

               BaseModel  Wins  Losses  WinRate_%
6   Baseline_Random Walk  7659   17252      30.75
22               XGBoost  6191    3067      66.87
1               Avg(B,G)  4058     520      88.64
2             Avg(B,G,L)  3947     520      88.36
14          Sta_GRU(B,G)  3864     544      87.66
15        Sta_GRU(B,G,L)  3661     556      86.82
3               Avg(B,L)  3646     609      85.69
18         Sta_LSTM(B,G)  3537    1147      75.51
19       Sta_LSTM(B,G,L)  3473     743      82.38
4               Avg(G,L)  3466     790      81.44
16          Sta_GRU(B,L)  3429     789      81.29
8                    GRU  3408     717      82.62
11        Sta_ANN(B,G,L)  3355    1288      72.26
5                   BRNN  3325     983      77.18
20         Sta_LSTM(B,L)  3300     804      80.41
13          Sta_ANN(G,L)  3298    1715      65.79
10          Sta_ANN(B,G)  3295    1125      74.55
21         Sta_LSTM(G,L)  3281     999      76.66
17          Sta_GRU(G,L)  3255    1143      74.01
